# Environment Setup
 
To run this notebook:
 
- Run the setup script to create the Python environment:
  `bash setup.sh`
 
- If you encounter issues, remove the environment and cache, then rerun:
  `rm -rf path/to/.venv /home/sagemaker-user/.cache && bash setup.sh`
 
- In JupyterHub, select the kernel named Python (Weather FM)
 
- Run the notebook from top to bottom  
  Only modify values in the configuration section
 
---
 
# Notebook Overview
 
This notebook generates visualizations of evaluation statistics for weather foundation models.
 
You will:
- Load postprocessed forecast outputs  
- Explore global and regional statistics  
- Visualize model performance across lead time, atmospheric levels, and variables  
 
All plots are saved to the specified output directory.
 
---
 
# Workflow Summary
 
- Import required libraries  
- Set configuration parameters  
- Load model statistics from storage  
- Separate data into global and regional statistics  
- Generate plots using provided functions  
 
---
 
# Configuration Parameters
 
This is the only section of the notebook that should be edited. These parameters control what data is loaded and what plots are produced.
 
Common parameters include:
 
- Start month 
  Which month in 2024 (chosen from "may" or "dec") to create plots for.
 
- Forecast model
  Identifier for the forecast model being evaluated; "Prithvi" or "AIFS".
 
- Variables  
  Variables to include in plots. See section below for a full list of variables (you can choose any combination of them!)
 
- Pressure levels  
  Pressure levels in hPa to create plots for (only affects 3D variables).
 
- Lead times  
  Forecast lead times in hours to use for plotting.
 
Participants should focus on selecting a model, variables, regions, and lead times relevant to their analysis.
 
---
 
# Key Concepts
 
## Latitude and Longitude
Define spatial location on Earth.  
Data is coarsened to a 91 x 144 grid for efficiency.
 
## Lead Time
Forecast horizon in hours.  
Example: 24 hours corresponds to a one-day forecast.
 
## Pressure Level
Represents vertical position in the atmosphere.  
Values near 1000 hPa are close to the surface, while lower values correspond to higher altitudes.
 
## Variables
 
3D variables are evaluated across multiple pressure levels:
- T temperature in Kelvin  
- Q specific humidity in g/kg  
- U east-west wind in m/s  
- V north-south wind in m/s  
- Z geopotential in m^2/s^2  
 
2D variables are single-level fields:
- T2m temperature at 2 meters  
- U10m and V10m wind at 10 meters  
- D2m dew point at 2 meters
- P sea level pressure
- PS surface pressure
 
---
 
# Model Comparisons
 
Statistics compare the forecast model against two references.
 
## Reanalysis datasets
Reanalysis products such as ERA5 combine observations with a physical model using data assimilation.  
They provide a physically consistent, observation-constrained estimate of the atmospheric state and are commonly used as a proxy for ground truth.
 
## Baseline models
Operational or analysis-driven models such as GEOS-FP are used as a baseline for comparison.  
These provide a strong reference point for evaluating whether a foundation model is competitive.
 
## Climatology
Climatology represents the long-term mean state of the atmosphere.  
It serves as a minimal-skill baseline that useful forecasts should outperform.
 
---
 
# Global Statistics
 
Global statistics are shown as spatial heatmaps.
 
Each figure includes:
- Row 1: forecast model compared to reanalysis  
- Row 2: forecast model compared to the baseline model  
 
## Metrics
 
- Forecast difference  
  Difference between forecast and reference  
  Values near zero indicate good agreement  
  Negative values indicate underprediction  
  Positive values indicate overprediction  
 
- Anomaly correlation coefficient (ACC)  
  Measures how well spatial patterns match between forecast and reference  
  Values closer to 1 indicate better agreement  
 
- Root mean squared error (RMSE)  
  Measures the magnitude of error  
  Lower values indicate better performance  
 
---
 
# Regional Statistics
 
Regional statistics are shown as time series over forecast lead time.
 
- Typically evaluated over a 7-day period  
- Compare the forecast model with the baseline model  
- Metrics include ACC and RMSE  
 
These plots show how forecast skill evolves over time and varies by region.
 
---
 
# Area-Weighted Metrics
 
All ACC and RMSE values used in this notebook are area-weighted.
 
- Grid cells at higher latitudes represent smaller physical areas than those near the equator  
- Without weighting, high-latitude regions would be overrepresented in global metrics  
 
Area-weighting accounts for this by applying latitude-based weights when computing statistics.
 
- Area-weighted ACC  
  Correlation computed with spatial weights proportional to grid cell area  
 
- Area-weighted RMSE  
  Error magnitude computed with the same spatial weighting  
 
These weighted metrics are computed during the postprocessing and statistics generation stage, not within this notebook.
 
---
 
# Notes for This Session
 
- Focus on interpreting patterns rather than individual values  
- Compare performance across regions, variables, and lead times  
- Use both ACC and RMSE together to understand model behavior  
 
Participants are encouraged to modify configuration settings to explore how model performance varies under different conditions.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import xarray as xr
from glob import glob
from pathlib import Path
import yaml
from tqdm import tqdm
import sys
import os
from datetime import datetime

sys.path.insert(0, "Plots")

# Import refactored plotting functions
from plots.load_stats import load_regional_stats_data, load_global_stats_data
from plots.plots_regional import create_regional_plots
from plots.plots_global import create_global_plots
from plots.utils import get_stat_dataset_filenames, create_base_patterns

## Config
This section can be configured to change the plots you generate! <mark>Pay close attention to the set of values presented, as if you pick a value outside of this range, the script will fail, as our data is only computed for these value ranges.</mark>

**MONTH:** 

Either "may" or "dec" (for May or Dec 2024).

**LEVELS:** 

Choose any combination from 

    [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50] hPa

**LEAD_TIMES:** 

Choose any combination from 
   
    [24, 48, 72, 96, 120, 144, 168].

**VARIABLES:** 

Choose any combination from among 2D and 3D variables:

    3D variables: ["Z", "Q", "T", "U", "V"] 

    2D variables: ["P", "PS", "D2m", "T2m", "V10m", "U10m"]



In [ ]:
# Stats are calculated for the full month of May/Dec 2024
# Month is not case-sensitive
MONTH = "Dec"

# Will only create level plots using these levels (zonal plots will still be created with all levels)
LEVELS = [925, 850, 500, 200]

# We will look for stats in this directory; stats are stored in netcdf files
FORECAST_MODEL = "AIFS"  # Choose from 'AIFS' or 'Prithvi'

# Will only plot these variables across 3D and 2D collections
VARIABLES = ["Z", "PS"]

# Will only plot leads from this list; forecasts were made for up to 7 days
# Leads are in hours
LEAD_TIMES = [24, 72, 120, 168]

## Get filenames of statistics netCDFs
These have already been created.

In [ ]:
# Create list of all statistics filenames across all experiments
exp_patterns = create_base_patterns(FORECAST_MODEL)
all_stat_filenames = get_stat_dataset_filenames(exp_patterns, MONTH)
regional_stat_filenames = all_stat_filenames["regional"]
global_stat_filenames = all_stat_filenames["global"]

# There should be 1 file per experiment in each of regional, global filename lists
regional_file_str = "\n".join(map(str, regional_stat_filenames))
global_file_str = "\n".join(map(str, global_stat_filenames))
print(regional_file_str)
print(global_file_str)

## Load stats data from the cloud

### Regional

In [ ]:
regional_data = load_regional_stats_data(regional_stat_filenames, vars_to_fetch=VARIABLES)

### Global

In [ ]:
global_data = load_global_stats_data(global_stat_filenames, vars_to_fetch=VARIABLES)

# Stat Plotting

## Make output directory
This is where the plots will go, separated into global and regional plots. 

In [ ]:
output_dir = Path(f"outputs/{FORECAST_MODEL}")
output_dir.mkdir(exist_ok=True, parents=True)

## Regional stat plotting

In [ ]:
# Creates only per-level plots of ACC/RMSE
create_regional_plots(
    regional_data,
    vars_to_plot=VARIABLES,
    levels_to_plot=LEVELS,
    plots_dir=output_dir
)

## Global stat plotting

In [ ]:
create_global_plots(
    global_data,
    vars_to_plot=VARIABLES,
    levels_to_plot=LEVELS,
    leads_to_plot=LEAD_TIMES,
    output_dir=output_dir
)